# Privacy and Governance Assessment- Credit Application Dataset
## 1. Introduction 

This notebook evaluates the credit application dataset from a data governance and privacy perspective. The analysis focuses on identifying potential regulatory risks associated with the processing of personal data and the use of automated decision-making in credit approval.

Credit scoring systems process large amounts of sensitive information about individuals, including demographic attributes, financial indicators, and behavioral data. When such information is used to support automated lending decisions, strong governance mechanisms are required to ensure compliance with data protection regulations and fairness principles.

The objective of this analysis is to assess whether the dataset used in NovaCred’s credit decision process presents potential privacy risks or governance gaps. In particular, the analysis focuses on three key areas:

1. **Identification of personally identifiable information (PII)** contained in the dataset.
2. **Evaluation of re-identification risks**, including the possibility of identifying individuals through combinations of quasi-identifiers.
3. **Assessment of governance and regulatory implications**, with particular attention to GDPR requirements and responsible data practices.

The goal is not only to identify potential privacy vulnerabilities but also to propose governance controls that can reduce regulatory and ethical risks in automated credit decision systems.

## 2. Dataset Overview
The dataset used in this analysis contains credit applications submitted to NovaCred’s lending platform. Each record represents an individual application and includes demographic information about the applicant, financial indicators, spending behavior, and the final credit decision.

The dataset is structured in several logical sections:

- **Applicant information**, containing personal and demographic attributes.
- **Financial data**, describing income and credit-related indicators.
- **Spending behavior**, capturing the applicant’s expenditure patterns.
- **Loan decision**, indicating whether the credit request was approved and under which conditions.

Understanding the structure of the dataset is an essential first step in assessing privacy risks and governance implications. Attributes related to personal identity, demographic characteristics, and geographic location may represent sensitive data and require careful handling under data protection regulations.

In [53]:
# 1. Import libraries 
import pandas as pd
import json
import hashlib

# 2. Load the cleaned dataset
with open('../data/processed/cleaned_credit_applications.json', 'r') as f:
    data = json.load(f)

# 3. Convert the data to a DataFrame:
df = pd.DataFrame(data)

# 4. Display:
df.head(3)

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,...,spending_Fitness,spending_Gambling,spending_Groceries,spending_Healthcare,spending_Insurance,spending_Rent,spending_Shopping,spending_Transportation,spending_Travel,spending_Utilities
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000.0,...,0,0,0,0,0,790,480,0,0,0
1,app_037,None,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032,78000.0,...,0,0,0,243,0,608,0,0,0,0
2,app_215,None,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000.0,...,0,0,0,0,0,109,0,0,0,0


In [54]:
# 5. Display all column names to understand the dataset schema
df.columns

Index(['_id', 'processing_timestamp', 'applicant_info.full_name',
       'applicant_info.email', 'applicant_info.ssn',
       'applicant_info.ip_address', 'applicant_info.gender',
       'applicant_info.date_of_birth', 'applicant_info.zip_code',
       'financials.annual_income', 'financials.credit_history_months',
       'financials.debt_to_income', 'financials.savings_balance',
       'decision.loan_approved', 'decision.rejection_reason', 'loan_purpose',
       'decision.interest_rate', 'decision.approved_amount', 'notes',
       'spending_Adult Entertainment', 'spending_Alcohol', 'spending_Dining',
       'spending_Education', 'spending_Entertainment', 'spending_Fitness',
       'spending_Gambling', 'spending_Groceries', 'spending_Healthcare',
       'spending_Insurance', 'spending_Rent', 'spending_Shopping',
       'spending_Transportation', 'spending_Travel', 'spending_Utilities'],
      dtype='object')

In [55]:
# 6. Show basic dataset info to check data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 34 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   _id                               500 non-null    object 
 1   processing_timestamp              62 non-null     object 
 2   applicant_info.full_name          500 non-null    object 
 3   applicant_info.email              500 non-null    object 
 4   applicant_info.ssn                496 non-null    object 
 5   applicant_info.ip_address         496 non-null    object 
 6   applicant_info.gender             500 non-null    object 
 7   applicant_info.date_of_birth      496 non-null    object 
 8   applicant_info.zip_code           500 non-null    object 
 9   financials.annual_income          500 non-null    float64
 10  financials.credit_history_months  500 non-null    float64
 11  financials.debt_to_income         500 non-null    float64
 12  financia

In [56]:
## 7. Display summary statistics for numeric columns to understand data distribution
df.describe()

,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.interest_rate,decision.approved_amount,spending_Adult Entertainment,spending_Alcohol,spending_Dining,spending_Education,...,spending_Fitness,spending_Gambling,spending_Groceries,spending_Healthcare,spending_Insurance,spending_Rent,spending_Shopping,spending_Transportation,spending_Travel,spending_Utilities
count,500.000000,500.000,500.000000,500.000000,292.000000,292.000000,500.000000,500.000000,500.000000,500.000000,...,500.000000,500.000000,500.00000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,82558.865578,50.666,0.245520,29644.332000,4.564726,47845.890411,5.912000,10.496000,62.408000,63.956000,...,64.664000,6.400000,63.41600,61.386000,64.352000,61.780000,49.288000,53.562000,78.944000,75.788000
std,28102.931999,31.080,0.136148,16674.262153,1.162866,18103.754530,61.570334,76.731411,184.030312,190.210584,...,185.131094,59.996927,185.76277,180.028012,183.320574,187.369192,161.110279,166.409057,206.771463,203.027928
min,0.000000,0.000,0.050000,0.000000,2.500000,15000.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,63000.000000,28.000,0.150000,17444.000000,3.500000,34000.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,81000.000000,49.000,0.230000,27401.000000,4.550000,48000.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,101000.000000,72.000,0.342500,38281.750000,5.600000,62250.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,171000.000000,133.000,1.850000,88078.000000,6.500000,80000.000000,848.000000,757.000000,936.000000,889.000000,...,1152.000000,751.000000,885.00000,897.000000,890.000000,897.000000,887.000000,899.000000,888.000000,881.000000


### Interpretation of Summary Statistics

The summary statistics provide an overview of the numerical variables contained in the dataset, including financial indicators and spending behavior attributes.

Several observations emerge from this initial exploration:

1) **Income variability**: Annual income shows a wide distribution, ranging from very low values to over 170,000. This variability is expected in credit datasets but may introduce potential bias in lending decisions if income becomes the dominant factor in approval outcomes.

2) **Debt-to-income ratio**: The debt-to-income ratio ranges approximately from 0.05 to 1.85. Very high values may indicate applicants with substantial financial risk, which may strongly influence loan approval decisions. Monitoring how this variable interacts with demographic attributes is important when evaluating potential discrimination.

3) **Credit history**: Credit history months range from 0 to over 130 months, indicating that the dataset includes both individuals with little credit experience and long-established borrowers. Applicants with very limited credit history may face systematic disadvantages in automated credit scoring systems.

4) **Spending behavior variables**: Spending-related attributes (e.g., groceries, entertainment, insurance, utilities) show large variation across applicants. While these variables may provide useful behavioral signals, they may also introduce indirect socioeconomic proxies that could correlate with protected attributes.

Of course, **Interest rate** (decision.interest_rate) and **approved loan amount** (decision.approved_amount) appear only for approved applications, as indicated by the lower number of observations compared to the total dataset size. This suggests that rejected applications do not receive these attributes, which is consistent with the decision process but should be considered when analyzing outcome distributions.

Overall, these descriptive statistics help identify key financial variables that may influence credit decisions and should therefore be monitored when evaluating potential bias and governance risks.

## 3. Identification of Personal Data (PII)

A key step in a governance and privacy assessment is identifying which attributes in the dataset qualify as personal data.

Under the GDPR, personal data includes any information that can directly or indirectly identify an individual. In credit application datasets, several fields typically contain sensitive personal information that requires careful handling and protection.

These attributes can be grouped into two main categories.

**Direct Identifiers**: Direct identifiers uniquely identify an individual and therefore represent the highest privacy risk. Examples include:
- Full name  
- Email address  
- Social Security Number (SSN)  
- IP address  

These attributes should normally be protected through mechanisms such as encryption or pseudonymization.

**Quasi-identifiers**:Quasi-identifiers do not uniquely identify individuals on their own but may allow re-identification when combined with other variables. Examples present in this dataset includes:
- Gender  
- Date of birth  
- ZIP code  

These attributes are particularly important when evaluating re-identification risks and applying anonymization techniques such as k-anonymity.

In [57]:
# 8. Define the columns that contain personally identifiable information (PII)
pii_columns = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.gender",
    "applicant_info.date_of_birth",
    "applicant_info.zip_code"
]

# 9. Display a preview of the personal data columns
df[pii_columns].head()

,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code
0,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036
1,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032
2,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075
3,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077
4,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080


In [58]:
# 10. Check missing values in PII columns
df[pii_columns].isnull().sum()

applicant_info.full_name        0
applicant_info.email            0
applicant_info.ssn              4
applicant_info.ip_address       4
applicant_info.gender           0
applicant_info.date_of_birth    4
applicant_info.zip_code         0
dtype: int64

In [59]:
# 11. Check number of unique values in key identifier fields
df[[
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn"
]].nunique()

applicant_info.full_name    475
applicant_info.email        494
applicant_info.ssn          494
dtype: int64

### Interpretation

The dataset contains several attributes that qualify as personally identifiable information (PII), including full name, email address, Social Security Number (SSN), IP address, gender, date of birth, and ZIP code. Many of these fields represent sensitive personal data and therefore require careful handling from a data governance and privacy perspective.

The preview of the dataset confirms that these attributes contain detailed personal information that could directly identify individuals. In particular, SSN, email address, and IP address represent strong identifiers and therefore pose a significant privacy risk if stored without appropriate protection mechanisms.

The missing value analysis shows that most personal data fields are complete. However, a small number of missing values are present in the SSN, IP address, and date of birth columns (four observations each). While the proportion of missing values is relatively small compared to the total dataset size, incomplete information in sensitive attributes may indicate inconsistencies in data collection or processing.

The uniqueness analysis highlights that email addresses and Social Security Numbers are almost entirely unique across the dataset. This suggests that these attributes function as reliable personal identifiers and could easily be used to identify individuals.

Full names, however, are not fully unique, as the number of unique names is slightly lower than the total number of records. This may reflect situations in which multiple records share the same name or cases where individuals submit more than one application. Based on the available information, it is not possible to determine the exact reason, but the result confirms that names alone are not reliable unique identifiers.

Overall, the presence of multiple direct identifiers in the dataset increases the risk of re-identification and highlights the importance of implementing privacy-preserving techniques such as pseudonymization before using the data for analytical purposes.

## 4. Psudonymization: 
Given the presence of several direct identifiers in the dataset, it is important to apply privacy-preserving techniques before using the data for analytical or modeling purposes.

One commonly used approach in data governance is **pseudonymization**, which replaces direct identifiers with transformed values that cannot be easily traced back to the original individual without additional information.

Unlike full anonymization, pseudonymization preserves the ability to link records belonging to the same individual while reducing the risk of direct identification.

In this section, a pseudonymization technique is applied to the Social Security Number (SSN) field using a cryptographic hash function. This transformation ensures that the original identifier is no longer directly visible while maintaining a consistent representation of the identifier across records.

In [60]:
# 12. Define a function to hash sensitive identifiers
def hash_identifier(value):
    
    # 12.1. Convert the value to string and encode it
    value_str = str(value).encode()
    
    # 12.2. Apply SHA-256 hashing
    return hashlib.sha256(value_str).hexdigest()

# 13. Create a pseudonymized version of the SSN column
df["hashed_ssn"] = df["applicant_info.ssn"].apply(hash_identifier)

# 14. Display original SSN and hashed SSN for comparison
df[["applicant_info.ssn", "hashed_ssn"]].head()

,applicant_info.ssn,hashed_ssn
0,596-64-4340,2caf30528c21a10e1307b27f9dbbfc312f0c00d46b333e...
1,425-69-4784,2f7da45fefdcfb2c5b4f5b6f1465c22054c36e04fc77c1...
2,370-78-5178,db120edcee2366a48d6d77c2db8c64c5536b8dc3c3c524...
3,194-35-1833,c835719be02018987096d6e49529a24b1d7e7ab35c84b1...
4,480-41-2475,41c7de40dc49185886e6ecb37346ec9eabce16087b7508...


### Interpretation

The transformation replaces the original SSN values with cryptographic hash values. As a result, the original identifier is no longer directly visible in the dataset, reducing the risk of exposing highly sensitive personal information.

The hashed identifiers remain consistent for the same input values. This means that records belonging to the same individual can still be linked across the dataset without revealing the original identifier. This property allows analytical tasks to be performed while improving the protection of sensitive data.

However, pseudonymization does not fully eliminate privacy risks. Since the transformation is deterministic, the same SSN will always generate the same hash value. If the hashing method or original identifiers were known, it could still be theoretically possible to reconstruct or match identities.

For this reason, pseudonymization should be considered a privacy-enhancing technique rather than full anonymization. Additional safeguards, such as limiting access to sensitive attributes and applying further anonymization techniques, may still be required.

## 5. Risk Assessment 
The presence of multiple personal data attributes in the dataset raises several regulatory considerations under the General Data Protection Regulation (GDPR).

Under the GDPR, personal data refers to any information that can directly or indirectly identify an individual. Several attributes in the dataset fall under this definition, including full name, email address, Social Security Number (SSN), IP address, date of birth, and ZIP code.

The storage and processing of such information require compliance with key GDPR principles such as data minimization, purpose limitation, and protection against unauthorized access.

Additionally, the dataset contains information related to credit approval decisions. Automated credit decision systems are considered high-risk processing activities because they may significantly affect individuals, particularly when decisions such as loan approvals or rejections are involved.

For this reason, such systems may fall under **Article 22 of the GDPR**, which regulates automated decision-making and profiling. Organizations using automated credit scoring systems must ensure transparency, fairness, and the possibility of human oversight.

The following table summarizes potential governance risks identified in the dataset and their relation to GDPR principles.

| Issue | GDPR Principle |
|------|---------------|
| Storage of SSN | Data minimization |
| Storage of IP address | Personal data processing |
| Storage of date of birth | Re-identification risk |
| Automated credit decisions | Article 22 – Automated decision-making |

In addition to GDPR considerations, automated credit decision systems may also fall under the **EU AI Act high-risk AI system category**.

AI systems used to evaluate creditworthiness or determine access to financial services are classified as **high-risk systems**, as they may significantly impact individuals’ economic opportunities.

Under the AI Act, such systems must satisfy several governance requirements, including:
- risk management procedures
- data governance and quality controls
- transparency and documentation
- human oversight mechanisms
- monitoring for bias or discriminatory outcomes.

This highlights the importance of implementing both privacy safeguards and fairness monitoring mechanisms when deploying automated credit scoring models.

## 6. Re-identification Risk & K-Anonimity Analysis

Even when direct identifiers such as names or Social Security Numbers are removed or pseudonymized, individuals may still be identifiable through combinations of other attributes. These attributes are known as **quasi-identifiers**.

Quasi-identifiers are variables that do not uniquely identify individuals on their own but may enable identification when combined with other attributes.

In credit application datasets, common quasi-identifiers include:
- Age (or date of birth)
- Gender
- ZIP code

Research in data privacy has shown that combinations of demographic attributes can uniquely identify individuals in a dataset. For example, the combination of **age, gender, and ZIP code** may significantly reduce anonymity because many individuals share rare or unique attribute combinations.

For this reason, datasets containing demographic and geographic information must be evaluated for **re-identification risk**, particularly when they are used for analytical or machine learning purposes.

A common technique used to evaluate this risk is **k-anonymity**, which measures whether individuals in the dataset are indistinguishable from at least *k-1* other individuals with respect to the selected quasi-identifiers.

The next section evaluates whether the dataset satisfies k-anonymity conditions using combinations of demographic attributes.

In [61]:
# 15. Convert date of birth to datetime format
df["applicant_info.date_of_birth"] = pd.to_datetime(df["applicant_info.date_of_birth"])

# 16. Calculate age from date of birth
today = pd.Timestamp.today()
df["age"] = (today - df["applicant_info.date_of_birth"]).dt.days // 365

# 17. Check age distribution
df["age"].describe()

count    496.000000
mean      40.772177
std       10.958808
min       23.000000
25%       32.000000
50%       39.000000
75%       47.000000
max       67.000000
Name: age, dtype: float64

In [62]:
# 18. K- Anonomity Assessment:

    # 18.1 Group records by quasi-identifier:
k_groups = df.groupby(
    ["age", "applicant_info.gender", "applicant_info.zip_code"]
).size().reset_index(name="count")

    # 18.2 Display smallest groups:
k_groups.sort_values("count").head(10)

,age,applicant_info.gender,applicant_info.zip_code,count
0,23.0,Female,90257,1
322,44.0,Male,10066,1
321,44.0,Male,10057,1
320,44.0,Male,10025,1
319,44.0,Male,10020,1
318,44.0,Male,10003,1
317,44.0,Male,10002,1
316,44.0,Female,90293,1
315,44.0,Female,90284,1
314,44.0,Female,90254,1


In [63]:
    # 18.3 Find the minimum k value
k_groups["count"].min()

1

### Interpretation

The age distribution indicates that applicants in the dataset are primarily adults, with ages ranging from approximately 23 to 67 years and an average age of around 41. This range is consistent with typical credit application populations and does not indicate obvious data anomalies.

To evaluate re-identification risk, records were grouped using three quasi-identifiers: **age, gender, and ZIP code**. These attributes do not uniquely identify individuals on their own but may enable identification when combined.

The grouping results show that several combinations of these attributes appear only **once** in the dataset. The minimum group size observed is **k = 1**, meaning that at least some individuals are uniquely identifiable based on the selected quasi-identifiers.

This result indicates that the dataset **does not satisfy k-anonymity** for the attributes age, gender, and ZIP code. As a consequence, there is a potential risk that individuals could be re-identified if the dataset were combined with external information sources.

From a data governance perspective, this highlights the importance of applying additional privacy protection techniques such as generalization, suppression, or aggregation of quasi-identifiers before sharing or analyzing the data.

In [64]:
# 19. Example of improving k-anonymity through anonymization techniques

    # 19.1 Create age groups instead of exact age (generalization)
df["age_group"] = pd.cut(
    df["age"],
    bins=[20,30,40,50,60,70],
    labels=["20-29","30-39","40-49","50-59","60-69"]
)

    # 19.2 Mask ZIP code by keeping only the first three digits (partial masking)
df["zip_masked"] = df["applicant_info.zip_code"].astype(str).str[:3] + "**"

    # 19.3 Mask gender values (suppression)
df["gender_masked"] = "*"

    # 19.4 Display anonymized attributes to illustrate transformation
df[[
    "age",
    "age_group",
    "applicant_info.gender",
    "gender_masked",
    "applicant_info.zip_code",
    "zip_masked"
]].head(10)

,age,age_group,applicant_info.gender,gender_masked,applicant_info.zip_code,zip_masked
0,25.0,20-29,Male,*,10036,100**
1,33.0,30-39,Male,*,10032,100**
2,36.0,30-39,Male,*,10075,100**
3,42.0,40-49,Male,*,10077,100**
4,26.0,20-29,Male,*,10080,100**
5,44.0,40-49,Female,*,10019,100**
6,36.0,30-39,Male,*,10022,100**
7,34.0,30-39,Female,*,90223,902**
8,35.0,30-39,Male,*,10044,100**
9,36.0,30-39,Male,*,10080,100**


In [65]:
# 20. k-anonymity analysis using anonymized quasi-identifiers

    # 20.1 Group records using generalized attributes
k_groups_generalized = df.groupby(
    ["age_group","gender_masked","zip_masked"],
    observed=False
).size().reset_index(name="count")

    # 20.2 Display smallest equivalence groups
k_groups_generalized.sort_values("count").head(10)

    # 20.3 Compute minimum k value
min_k_generalized = k_groups_generalized["count"].min()
min_k_generalized

0

### Interpretation after Anonymization

To reduce re-identification risk, several anonymization techniques were applied to the quasi-identifiers:

- **Age generalization**, replacing exact ages with age ranges.
- **ZIP code masking**, keeping only the first three digits of the ZIP code.
- **Gender suppression**, replacing gender values with a generic symbol.

These transformations reduce the granularity of the original attributes and demonstrate how anonymization techniques can be applied to protect sensitive information.

However, the k-anonymity analysis shows that some combinations of the generalized attributes still appear only once in the dataset. This indicates that **k = 1**, meaning that certain individuals remain uniquely identifiable even after the applied transformations.

From a governance and privacy perspective, this result highlights that simple anonymization strategies may not always be sufficient to eliminate re-identification risk. Additional techniques such as stronger generalization, attribute suppression, or aggregation may be required before sharing or externally processing the dataset.

## 8. Privacy Risks Identified

Based on the analysis performed so far, the dataset presents several privacy risks. These risks can be grouped into **direct risks** (linked to explicit identifiers) and **indirect risks** (linked to re-identification through combinations of attributes).

### 8.1 Direct Privacy Risks (High Severity)

**Presence of direct identifiers**
The dataset contains direct identifiers such as full name, email address, SSN, and IP address. These fields can directly reveal an individual’s identity if accessed or leaked.

**Highly sensitive identifier (SSN)**
Social Security Number is a highly sensitive identifier. Its presence in the dataset significantly increases the impact of any data breach and raises strong concerns under GDPR principles such as data minimization and security of processing.

**IP address storage**
IP address is personal data under GDPR. If stored without clear necessity and safeguards, it increases exposure to privacy risks and may enable tracking or linkage.

### 8.2. Indirect Privacy Risks (Re-identification)

**Re-identification via quasi-identifiers**
Even after removing or pseudonymizing direct identifiers, demographic and geographic attributes (e.g., age, gender, ZIP code) may still enable identification when combined.  
The k-anonymity assessment showed that some quasi-identifier combinations appear only once in the dataset, indicating that re-identification risk remains present.

**Linkage risk with external datasets**
If an attacker has access to external information (e.g., public records or demographic statistics), quasi-identifiers could be used to match and re-identify individuals in the dataset.

**Contextual Risk (Automated Credit Decisions)**

**High-impact decisions**
The dataset contains information about credit approval outcomes. Since credit decisions may significantly affect individuals’ economic opportunities, the processing context increases the sensitivity of the dataset and strengthens governance requirements around privacy, transparency, and accountability.

## 9. Governance Gaps

Beyond the identification of privacy risks, the analysis of the dataset also reveals several potential governance gaps related to data management, decision transparency, and system accountability.

### 9.1. Lack of Clear Data Minimization Practices

Several highly sensitive attributes are stored in the dataset, including Social Security Number, IP address, and full date of birth. While these attributes may be useful for identity verification during the application process, storing them in analytical datasets may violate the **data minimization principle** under GDPR, which requires that only strictly necessary data be processed.

### 9.2. Weak Audit and Processing Traceability

The dataset contains a field called `processing_timestamp`, but most records appear to have missing values. This suggests that the system may not consistently record when applications are processed or decisions are generated.

From a governance perspective, missing processing timestamps reduce the ability to perform **audits**, track data processing activities, and ensure accountability in automated decision systems.

### 9.3. Limited Decision Transparency

Although the dataset includes a field indicating whether a loan was approved or rejected, the information explaining *why* a decision was made appears limited or incomplete. In automated decision systems, the absence of structured explanations may reduce transparency and make it difficult to justify outcomes to affected individuals.

This becomes particularly important in financial services, where individuals may request explanations for credit decisions.

### 9.4. Absence of Explicit Privacy Controls

The dataset structure does not show clear indicators of implemented privacy controls such as:

- consent tracking
- data retention policies
- anonymization status indicators
- access control metadata

Without these elements, it is difficult to verify whether the data processing lifecycle complies with privacy-by-design principles.

### 9.5 Lack of Fairness Monitoring Indicators

While the dataset contains demographic attributes such as gender and age, there is no evidence of fields or metrics dedicated to monitoring fairness or bias in credit approval decisions. In automated lending systems, the absence of such monitoring mechanisms may increase the risk of **undetected discriminatory outcomes**.

## 10. Governance Controls

To mitigate the privacy, fairness, and regulatory risks identified in the previous sections, several governance controls should be implemented when managing and analyzing credit application datasets.

### 10.1. Data Protection Controls

**Pseudonymization of direct identifiers**

Highly sensitive identifiers such as Social Security Numbers, email addresses, and full names should be pseudonymized or removed from analytical datasets. Direct identifiers should only be accessible in operational systems where they are strictly necessary for identity verification.

**Encryption of sensitive personal data**

Sensitive personal data such as SSN and IP addresses should be protected using strong encryption mechanisms both at rest and in transit. This reduces the impact of potential data breaches.

**Data minimization**

Attributes that are not strictly required for analytical purposes should be removed from the dataset. For example, storing full dates of birth or detailed geographic identifiers may increase re-identification risk without significantly improving model performance.

### 10.2. Privacy-Preserving Data Processing

**Anonymization of quasi-identifiers**

Techniques such as generalization and masking should be applied to quasi-identifiers. In this analysis, examples include:

- age generalization (age groups instead of exact age)
- ZIP code masking
- suppression of demographic attributes

These techniques help improve anonymity while preserving the usefulness of the dataset for analytical tasks.

**Regular re-identification risk assessments**

Organizations should periodically evaluate datasets using techniques such as **k-anonymity analysis** to detect whether individuals could be re-identified through combinations of quasi-identifiers.

### 10.3. Fairness and Bias Monitoring

**Bias detection mechanisms**

Credit decision systems should include monitoring procedures to evaluate whether approval outcomes differ significantly across demographic groups such as gender or age.

**Fairness metrics**

Metrics such as disparate impact, approval rate differences, or statistical parity should be monitored regularly to detect potential discrimination.

### 10.4. Human Oversight

Automated credit decision systems should incorporate mechanisms allowing **human review of decisions**, particularly in cases where applications are rejected. This helps ensure accountability and reduces the risk of unjust automated outcomes.

### 10.5. Auditability and Documentation

Organizations should maintain clear **audit logs** recording when data is processed and when automated decisions are generated. This includes maintaining processing timestamps and decision documentation to enable internal audits and regulatory inspections.

## 11. Regulatory Mapping

The analysis of the credit application dataset highlights several privacy, fairness, and governance risks that are relevant under European regulatory frameworks. In particular, the identified issues relate to the **General Data Protection Regulation (GDPR)** and the **EU Artificial Intelligence Act (AI Act)**.

The following table summarizes how the risks identified in the dataset relate to specific regulatory requirements.

| Identified Risk | Regulatory Framework | Relevant Principle or Article |
|-----------------|----------------------|--------------------------------|
| Storage of direct identifiers (SSN, email, IP address) | GDPR | Data minimization (Art. 5) |
| Re-identification risk through quasi-identifiers | GDPR | Privacy by design and by default (Art. 25) |
| Processing of sensitive personal data | GDPR | Security of processing (Art. 32) |
| Automated credit approval decisions | GDPR | Automated decision-making (Art. 22) |
| Potential demographic bias in approval outcomes | EU AI Act | High-risk AI systems (creditworthiness assessment) |
| Lack of fairness monitoring mechanisms | EU AI Act | Risk management and bias mitigation requirements |

Under the **EU AI Act**, AI systems used to evaluate creditworthiness or determine access to financial services are classified as **high-risk systems**. As a result, organizations deploying such systems must implement strict governance mechanisms, including risk management procedures, data quality controls, human oversight, and continuous monitoring.

The governance controls proposed in the previous section contribute to aligning the system with both **GDPR privacy principles** and **AI Act risk management requirements**.

## 12. Conclusion

This analysis evaluated the credit application dataset from a data governance, privacy, and regulatory compliance perspective. The objective was to identify potential risks related to the processing of personal data and the use of automated decision-making in credit approval systems.

The results show that the dataset contains several **direct identifiers**, including full name, email address, Social Security Number, and IP address. These attributes represent a significant privacy risk if not properly protected and highlight the importance of implementing strong data protection mechanisms such as pseudonymization and encryption.

In addition to direct identifiers, the dataset also contains several **quasi-identifiers**, including age, gender, and ZIP code. The k-anonymity analysis demonstrated that some combinations of these attributes appear only once in the dataset, indicating that individuals may still be re-identified even after removing or masking direct identifiers.

The analysis also revealed several **governance gaps**, including limited traceability of data processing activities, incomplete decision transparency, and the absence of explicit mechanisms for monitoring fairness and bias in credit approval outcomes.

To mitigate these risks, a set of **governance controls** was proposed. These include pseudonymization of sensitive identifiers, anonymization of quasi-identifiers, implementation of fairness monitoring mechanisms, human oversight of automated decisions, and improved auditability of data processing activities.

Finally, the analysis highlighted the relevance of both the **GDPR** and the **EU AI Act** in regulating automated credit decision systems. Creditworthiness assessment systems are considered **high-risk AI systems**, requiring strong governance frameworks to ensure transparency, fairness, and accountability.

Overall, this assessment demonstrates the importance of integrating privacy protection, fairness monitoring, and governance controls when developing and deploying data-driven credit decision systems.